# P3.09 Discovery: GPU Capability Probe

**PURPOSE:** Detect GPU availability, CUDA support, and vendor specifics

**TIMEOUT:** ≤5 minutes

**CRITICAL:** GPU is optional but nice-to-have for P3.09 shards. This probe documents what's available.

In [ ]:
import json
import os
import sys
import subprocess
import time
from datetime import datetime
from pathlib import Path

# Session metadata
DISCOVERY_SESSION = {
    "session_id": f"gpu_probe_{int(time.time())}",
    "timestamp": datetime.now().isoformat(),
    "timeout_hard": 300,
    "timeout_warning": 270,
    "notebook_name": "kaggle_discovery_01_gpu_capability_probe",
    "results": []
}

start_time = time.time()

def log_test(test_name, result, evidence, duration_s):
    """Log a discovery test result."""
    DISCOVERY_SESSION["results"].append({
        "test": test_name,
        "result": result,  # PASS, FAIL, UNKNOWN, BLOCKED
        "evidence": evidence,
        "duration_s": duration_s,
        "timestamp": datetime.now().isoformat()
    })
    print(f"[{result:8s}] {test_name} ({duration_s:.1f}s)")

def check_timeout():
    """Check if we're approaching timeout."""
    elapsed = time.time() - start_time
    if elapsed > DISCOVERY_SESSION["timeout_hard"]:
        raise RuntimeError(f"HARD TIMEOUT: {elapsed:.0f}s")
    elif elapsed > DISCOVERY_SESSION["timeout_warning"]:
        print(f"⚠️  WARNING: {DISCOVERY_SESSION['timeout_hard'] - elapsed:.0f}s remaining")
    return elapsed

print(f"🔬 P3.09 GPU Capability Discovery: {DISCOVERY_SESSION['session_id']}")
print(f"📍 Started: {DISCOVERY_SESSION['timestamp']}")

## Test 1: PyTorch CUDA Detection

In [ ]:
test_start = time.time()
check_timeout()

try:
    pytorch_probe = {
        "torch_installed": False,
        "cuda_available": False,
        "device_count": 0,
        "device_names": [],
        "torch_version": None
    }
    
    try:
        import torch
        pytorch_probe["torch_installed"] = True
        pytorch_probe["torch_version"] = torch.__version__
        pytorch_probe["cuda_available"] = torch.cuda.is_available()
        
        if pytorch_probe["cuda_available"]:
            pytorch_probe["device_count"] = torch.cuda.device_count()
            pytorch_probe["device_names"] = [
                torch.cuda.get_device_name(i) for i in range(pytorch_probe["device_count"])
            ]
            pytorch_probe["cuda_version"] = torch.version.cuda
    except ImportError:
        pass
    
    result_status = "PASS" if pytorch_probe["cuda_available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("pytorch_cuda_detection", result_status, pytorch_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("pytorch_cuda_detection", "FAIL", str(e), duration)

## Test 2: nvidia-smi Command Probe

In [ ]:
test_start = time.time()
check_timeout()

try:
    nvidia_probe = {
        "nvidia_smi_available": False,
        "gpu_detected": False,
        "output": None,
        "error": None
    }
    
    try:
        result = subprocess.run(
            ["nvidia-smi"],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            nvidia_probe["nvidia_smi_available"] = True
            nvidia_probe["output"] = result.stdout[:500]  # First 500 chars
            nvidia_probe["gpu_detected"] = "GPU" in result.stdout or "NVIDIA" in result.stdout
        else:
            nvidia_probe["error"] = result.stderr[:200]
    except FileNotFoundError:
        nvidia_probe["error"] = "nvidia-smi not found in PATH"
    except subprocess.TimeoutExpired:
        nvidia_probe["error"] = "nvidia-smi command timeout"
    
    result_status = "PASS" if nvidia_probe["nvidia_smi_available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("nvidia_smi_probe", result_status, nvidia_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("nvidia_smi_probe", "FAIL", str(e), duration)

## Test 3: TensorFlow GPU Probe

In [ ]:
test_start = time.time()
check_timeout()

try:
    tf_probe = {
        "tensorflow_installed": False,
        "gpu_available": False,
        "device_list": [],
        "tensorflow_version": None
    }
    
    try:
        import tensorflow as tf
        tf_probe["tensorflow_installed"] = True
        tf_probe["tensorflow_version"] = tf.__version__
        
        # List physical devices
        gpu_devices = tf.config.list_physical_devices('GPU')
        if len(gpu_devices) > 0:
            tf_probe["gpu_available"] = True
            tf_probe["device_list"] = [str(d) for d in gpu_devices]
    except ImportError:
        pass
    
    result_status = "PASS" if tf_probe["gpu_available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("tensorflow_gpu_probe", result_status, tf_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("tensorflow_gpu_probe", "FAIL", str(e), duration)

## Test 4: CPU Fallback Validation

In [ ]:
test_start = time.time()
check_timeout()

try:
    cpu_probe = {
        "cpu_cores_available": os.cpu_count(),
        "cpu_fallback_viable": False,
        "note": "CPU-only rendering is valid fallback even if GPU unavailable"
    }
    
    # CPU is ALWAYS available; Remotion can render on CPU
    cpu_probe["cpu_fallback_viable"] = cpu_probe["cpu_cores_available"] is not None and cpu_probe["cpu_cores_available"] >= 1
    
    result_status = "PASS" if cpu_probe["cpu_fallback_viable"] else "FAIL"
    duration = time.time() - test_start
    log_test("cpu_fallback_validation", result_status, cpu_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("cpu_fallback_validation", "FAIL", str(e), duration)

## Final Report

In [ ]:
# Calculate statistics
results = DISCOVERY_SESSION["results"]
passed = sum(1 for r in results if r["result"] == "PASS")
failed = sum(1 for r in results if r["result"] == "FAIL")
unknown = sum(1 for r in results if r["result"] == "UNKNOWN")
blocked = sum(1 for r in results if r["result"] == "BLOCKED")

total_time = time.time() - start_time

DISCOVERY_SESSION.update({
    "summary": {
        "total_tests": len(results),
        "passed": passed,
        "failed": failed,
        "unknown": unknown,
        "blocked": blocked,
        "total_time_s": total_time,
        "verdict": "GPU optional; CPU fallback always viable"
    }
})

# Write artifact
output_path = Path("/kaggle/working/discovery_gpu_probe_results.json")
output_path.write_text(json.dumps(DISCOVERY_SESSION, indent=2))

print(f"\n📊 GPU Capability Summary:")
print(f"   PASS:    {passed}")
print(f"   FAIL:    {failed}")
print(f"   UNKNOWN: {unknown}")
print(f"   BLOCKED: {blocked}")
print(f"   Total:   {len(results)} tests in {total_time:.1f}s")
print(f"\n✅ Results saved to: {output_path}")